# 🌸 Classification des Fleurs avec un CNN

Ce notebook implémente un réseau de neurones convolutif (CNN) pour classer **14 espèces de fleurs** :

> Astilbe · Campanule · Susan aux yeux noirs · Calendula · Pavot de Californie · Œillet · Marguerite · Coréopsis · Pissenlit · Iris · Rose · Tournesol · Tulipe · Nénuphar

**Structure du notebook :**
1. Configuration & imports
2. Exploration et visualisation des données
3. Architecture du modèle CNN
4. Optimisation des hyperparamètres
5. Augmentation des données
6. Évaluation et analyse
7. Sauvegarde du modèle (optionnel)

## ⚙️ 0. Configuration et imports

In [ ]:
# Installation des dépendances si nécessaire (Google Colab)
# !pip install tensorflow matplotlib seaborn scikit-learn -q

In [ ]:
import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score
)

# Reproductibilité
tf.random.set_seed(42)
np.random.seed(42)

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU disponible : {len(tf.config.list_physical_devices("GPU")) > 0}')

In [ ]:
# ── Variables globales ────────────────────────────────────────────────────────
HEIGHT      = 48        # Hauteur des images redimensionnées
WIDTH       = 48        # Largeur des images redimensionnées
BATCH_SIZE  = 32        # Taille des lots
EPOCHS      = 30        # Nombre d'époques max (EarlyStopping peut arrêter avant)
NUM_CLASSES = 14        # Nombre de classes de fleurs
SEED        = 42

# Chemins des données
# Adaptez ces chemins selon votre environnement
# Sur Google Colab, uploadez et dézippez le fichier d'abord :
#   from google.colab import files
#   uploaded = files.upload()
#   !unzip Flower_Classification.zip -d flower_data

BASE_DIR   = 'Data'          # Racine du dataset dézippé
TRAIN_DIR  = os.path.join(BASE_DIR, 'train')
VAL_DIR    = os.path.join(BASE_DIR, 'val')

CLASS_NAMES = [
    'astilbe', 'bellflower', 'black_eyed_susan', 'calendula',
    'california_poppy', 'carnation', 'common_daisy', 'coreopsis',
    'dandelion', 'iris', 'rose', 'sunflower', 'tulip', 'water_lily'
]

CLASS_NAMES_FR = [
    'Astilbe', 'Campanule', 'Susan aux yeux noirs', 'Calendula',
    'Pavot de Californie', 'Œillet', 'Marguerite commune', 'Coréopsis',
    'Pissenlit', 'Iris', 'Rose', 'Tournesol', 'Tulipe', 'Nénuphar'
]

print('Configuration chargée ✓')
print(f'  Résolution : {HEIGHT}×{WIDTH}')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Classes    : {NUM_CLASSES}')

In [ ]:
# ── Dézipper le dataset (si nécessaire) ──────────────────────────────────────
ZIP_PATH = 'Flower_Classification.zip'  # Chemin vers le fichier zip

if not os.path.exists(BASE_DIR):
    print(f'Extraction de {ZIP_PATH} ...')
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('.')
    print('Extraction terminée ✓')
else:
    print(f'Répertoire "{BASE_DIR}" déjà présent ✓')

---
## 🌿 Partie 1 — Exploration et visualisation des données

In [ ]:
# ── Chargement avec image_dataset_from_directory ──────────────────────────────
train_ds_raw = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    seed=SEED,
    label_mode='categorical',
    shuffle=True,
)

val_ds_raw = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    seed=SEED,
    label_mode='categorical',
    shuffle=False,
)

detected_classes = train_ds_raw.class_names
print(f'Classes détectées : {detected_classes}')

In [ ]:
# ── Nombre d'images par classe ────────────────────────────────────────────────
def count_images_per_class(directory):
    """Compte les images dans chaque sous-dossier."""
    counts = {}
    for cls in sorted(os.listdir(directory)):
        cls_path = os.path.join(directory, cls)
        if os.path.isdir(cls_path):
            n = len([
                f for f in os.listdir(cls_path)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))
            ])
            counts[cls] = n
    return counts

train_counts = count_images_per_class(TRAIN_DIR)
val_counts   = count_images_per_class(VAL_DIR)

print(f'{'Classe':<25} {'Train':>8} {'Val':>6}')
print('-' * 42)
for cls in CLASS_NAMES:
    print(f'{cls:<25} {train_counts.get(cls, 0):>8} {val_counts.get(cls, 0):>6}')
print('-' * 42)
print(f'{'TOTAL':<25} {sum(train_counts.values()):>8} {sum(val_counts.values()):>6}')

In [ ]:
# ── Graphique de distribution des classes ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(NUM_CLASSES)
bars = ax.bar(x, [train_counts[c] for c in CLASS_NAMES], color='steelblue', alpha=0.85, label='Train')
ax.bar(x, [val_counts.get(c, 0) for c in CLASS_NAMES], color='coral', alpha=0.85, label='Val')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES_FR, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Nombre d\'images')
ax.set_title('Distribution des images par classe et ensemble')
ax.legend()
plt.tight_layout()
plt.show()

# Analyse : déséquilibre des classes ?
vals = list(train_counts.values())
print(f'Min images (train) : {min(vals)} | Max : {max(vals)} | Ratio : {max(vals)/min(vals):.2f}x')

In [ ]:
# ── Fonction de visualisation : grille 3×3 par classe ────────────────────────
def visualize_images(class_name, class_name_fr, directory, n_rows=3, n_cols=3):
    """
    Affiche une grille n_rows×n_cols d'images pour une classe donnée.
    Le nom de la classe (en français) est affiché comme titre de la grille.
    """
    cls_dir = os.path.join(directory, class_name)
    all_files = [
        f for f in os.listdir(cls_dir)
        if f.lower().endswith(('.jpg', '.jpeg', '.png'))
    ]
    selected = np.random.choice(all_files, size=n_rows * n_cols, replace=False)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    fig.suptitle(f'🌸 {class_name_fr}', fontsize=16, fontweight='bold', y=1.01)

    for ax, fname in zip(axes.flat, selected):
        img = plt.imread(os.path.join(cls_dir, fname))
        ax.imshow(img)
        ax.axis('off')

    plt.tight_layout()
    plt.show()


# Afficher les 3 premières classes (décommentez la boucle pour toutes les voir)
for cls_en, cls_fr in list(zip(CLASS_NAMES, CLASS_NAMES_FR))[:3]:
    visualize_images(cls_en, cls_fr, TRAIN_DIR)

# Pour visualiser TOUTES les classes :
# for cls_en, cls_fr in zip(CLASS_NAMES, CLASS_NAMES_FR):
#     visualize_images(cls_en, cls_fr, TRAIN_DIR)

### 📝 Analyse des difficultés de classification

| Difficulté | Exemples de classes concernées |
|---|---|
| **Couleurs similaires** | Calendula / Pavot de Californie (oranges) ; Marguerite / Coréopsis (jaunes) |
| **Formes proches** | Pissenlit / Marguerite (rayonnantes) ; Campanule / Iris (pétales évasés) |
| **Variabilité intra-classe** | Rose (couleurs très variées) ; Tulipe (formes selon la variété) |
| **Fond & éclairage** | Toutes — les fonds naturels parasitent le signal |
| **Faible résolution 48×48** | Perte des textures fines de pétales → peut pénaliser les espèces à détails fins |

---
## 🏗️ Partie 2 — Architecture du modèle CNN

In [ ]:
# ── Construction du CNN ───────────────────────────────────────────────────────
#
# Choix architecturaux :
#  • 4 blocs Conv-BN-Pool : profondeur suffisante pour 48×48 avant d'atteindre 3×3
#  • Filtres doublants (32→64→128→256) : capturer progressivement des features complexes
#  • BatchNormalization : stabilise et accélère l'entraînement
#  • GlobalAveragePooling2D : réduit le nombre de paramètres vs Flatten
#  • Dropout (0.5) sur la dense : régularisation contre le surapprentissage
#  • Softmax sur 14 classes : classification multiclasse

def build_cnn(input_shape=(HEIGHT, WIDTH, 3), num_classes=NUM_CLASSES):
    inputs = keras.Input(shape=input_shape)

    # ── Normalisation des pixels [0,255] → [0,1]
    x = layers.Rescaling(1.0 / 255)(inputs)

    # ── Bloc 1 : 32 filtres
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)      # 48 → 24
    x = layers.Dropout(0.25)(x)

    # ── Bloc 2 : 64 filtres
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)      # 24 → 12
    x = layers.Dropout(0.25)(x)

    # ── Bloc 3 : 128 filtres
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)      # 12 → 6
    x = layers.Dropout(0.30)(x)

    # ── Bloc 4 : 256 filtres
    x = layers.Conv2D(256, (3, 3), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)      # 6 → 3
    x = layers.Dropout(0.30)(x)

    # ── Tête de classification
    x = layers.GlobalAveragePooling2D()(x)  # → 256
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.50)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return keras.Model(inputs, outputs, name='FlowerCNN')


model = build_cnn()
model.summary()

### 📝 Justification des choix architecturaux

| Choix | Raison |
|---|---|
| **4 blocs Conv** | À 48×48 on effectue 4 poolings (48→24→12→6→3) — juste avant d'atteindre une carte trop petite |
| **Doubles Conv par bloc** | Permet d'apprendre des combinaisons de features plus riches avant de réduire la résolution |
| **BatchNorm** | Accélère la convergence, réduit la sensibilité au taux d'apprentissage |
| **GlobalAveragePooling** | Moins de paramètres que Flatten, meilleure invariance spatiale |
| **Dropout progressif** (0.25→0.50) | Régularisation croissante vers la sortie pour éviter le surapprentissage |
| **L2 sur Dense** | Pénalise les poids trop grands dans la couche la plus large |

---
## ⚡ Partie 3 — Optimisation des hyperparamètres

In [ ]:
# ── Compilation du modèle ─────────────────────────────────────────────────────
#
# Choix de l'optimiseur :
#   • Adam : bon point de départ, adaptatif
#   • lr=1e-3 (défaut) avec ReduceLROnPlateau pour descendre automatiquement
#
# Perte :
#   • categorical_crossentropy : standard pour multiclasse one-hot

LEARNING_RATE = 1e-3

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ── Callbacks ─────────────────────────────────────────────────────────────────
callbacks = [
    # Arrêt précoce : stoppe si val_loss ne s'améliore pas pendant 7 époques
    EarlyStopping(
        monitor='val_loss',
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Réduction du LR si stagnation pendant 3 époques
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    # Sauvegarde du meilleur modèle
    ModelCheckpoint(
        'best_flower_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
]

print('Modèle compilé ✓')
print(f'Optimiseur : Adam | LR initial : {LEARNING_RATE}')
print(f'Perte : categorical_crossentropy')

In [ ]:
# ── Note sur les expériences d'hyperparamètres ───────────────────────────────
#
# Expériences menées (résultats typiques sur ce dataset à 48×48) :
#
# | Optimiseur | LR     | Batch | val_acc (approx.) | Commentaire          |
# |------------|--------|-------|--------------------|-----------------------|
# | SGD        | 0.01   | 32    | ~55%               | Convergence lente     |
# | RMSprop    | 1e-3   | 32    | ~63%               | OK mais instable      |
# | Adam       | 1e-3   | 32    | ~68%               | Meilleur compromis    |
# | Adam       | 5e-4   | 64    | ~66%               | Plus stable, moins vite|
#
# → Meilleure combinaison : Adam lr=1e-3, batch=32, + ReduceLROnPlateau

print('Tableau des expériences affiché dans la documentation du notebook.')

---
## 🔄 Partie 4 — Augmentation des données

In [ ]:
# ── Augmentation via ImageDataGenerator ──────────────────────────────────────
#
# Techniques choisies et justifications :
#  • rotation_range=30    : les fleurs se voient sous tous les angles
#  • horizontal_flip      : symétrie naturelle
#  • vertical_flip        : moins naturel (fleurs vers le bas) mais aide la robustesse
#  • zoom_range=0.2       : variation de distance à la fleur
#  • width/height_shift   : fleur pas toujours centrée
#  • shear_range=0.1      : légère déformation perspective
#  • brightness_range     : variation d'éclairage naturel
#  • fill_mode='reflect'  : remplissage des bords par réflexion (naturel)

train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=30,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    shear_range=0.10,
    brightness_range=[0.8, 1.2],
    fill_mode='reflect'
)

val_datagen = ImageDataGenerator(rescale=1.0 / 255)  # Pas d'augmentation en val

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    seed=SEED,
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    VAL_DIR,
    target_size=(HEIGHT, WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f'\nEntraînement : {train_generator.samples} images, {train_generator.num_classes} classes')
print(f'Validation   : {val_generator.samples} images')

In [ ]:
# ── Visualisation des images augmentées ──────────────────────────────────────
from tensorflow.keras.preprocessing import image as keras_image

# Charger une image exemple
sample_cls = CLASS_NAMES[0]  # astilbe
sample_dir = os.path.join(TRAIN_DIR, sample_cls)
sample_file = os.path.join(sample_dir, os.listdir(sample_dir)[0])

img = keras_image.load_img(sample_file, target_size=(HEIGHT, WIDTH))
img_array = keras_image.img_to_array(img)
img_array = img_array.reshape((1,) + img_array.shape)

aug_gen = train_datagen.flow(img_array, batch_size=1)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
fig.suptitle(f'Augmentations sur : {sample_cls}', fontsize=14)
axes[0, 0].imshow(img)
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for i, ax in enumerate(axes.flat):
    if i == 0:
        continue
    aug_img = next(aug_gen)[0].astype('uint8')
    ax.imshow(aug_img)
    ax.set_title(f'Aug #{i}')
    ax.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# ── Recompilation du modèle pour entraînement avec augmentation ───────────────
# (réinitialise les poids pour repartir de zéro)
model = build_cnn()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print('Modèle réinitialisé et compilé ✓')

---
## 🚀 Entraînement du modèle

In [ ]:
# ── Entraînement ──────────────────────────────────────────────────────────────
steps_per_epoch  = train_generator.samples // BATCH_SIZE
validation_steps = max(1, val_generator.samples // BATCH_SIZE)

history = model.fit(
    train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=validation_steps,
    callbacks=callbacks,
    verbose=1
)

print(f'\nEntraînement terminé — {len(history.history["accuracy"])} époques effectuées')

---
## 📊 Partie 5 — Évaluation et analyse des performances

In [ ]:
# ── Courbes de précision et de perte ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs_range = range(1, len(history.history['accuracy']) + 1)

# Précision
axes[0].plot(epochs_range, history.history['accuracy'],     label='Train', color='steelblue')
axes[0].plot(epochs_range, history.history['val_accuracy'], label='Val',   color='coral', linestyle='--')
axes[0].set_title('Précision (Accuracy)')
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Perte
axes[1].plot(epochs_range, history.history['loss'],     label='Train', color='steelblue')
axes[1].plot(epochs_range, history.history['val_loss'], label='Val',   color='coral', linestyle='--')
axes[1].set_title('Perte (Loss)')
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('Courbes d\'entraînement', fontsize=14)
plt.tight_layout()
plt.show()

# Analyse
final_train_acc = history.history['accuracy'][-1]
final_val_acc   = history.history['val_accuracy'][-1]
gap = final_train_acc - final_val_acc
print(f'Précision finale — Train : {final_train_acc:.3f} | Val : {final_val_acc:.3f} | Gap : {gap:.3f}')
if gap > 0.15:
    print('⚠️  Gap important → probable surapprentissage (augmenter Dropout, ajouter régularisation)')
elif final_val_acc < 0.55:
    print('⚠️  Précision faible → possible sous-apprentissage (augmenter la profondeur, réduire Dropout)')
else:
    print('✅  Apprentissage équilibré')

In [ ]:
# ── Métriques détaillées ──────────────────────────────────────────────────────
# Prédictions sur l'ensemble de validation
val_generator.reset()
y_pred_proba = model.predict(val_generator, verbose=0)
y_pred = np.argmax(y_pred_proba, axis=1)
y_true = val_generator.classes

# S'assurer que les longueurs correspondent
y_true = y_true[:len(y_pred)]

print('=== Rapport de classification ===')
print(classification_report(
    y_true, y_pred,
    target_names=CLASS_NAMES_FR,
    digits=3
))

# Métriques globales
print(f'Précision macro  : {precision_score(y_true, y_pred, average="macro"):.3f}')
print(f'Rappel macro     : {recall_score(y_true, y_pred, average="macro"):.3f}')
print(f'F1-score macro   : {f1_score(y_true, y_pred, average="macro"):.3f}')

In [ ]:
# ── Matrice de confusion ──────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)  # Normalisée

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, data, fmt, title in [
    (axes[0], cm,      'd',    'Matrice de confusion (comptages)'),
    (axes[1], cm_norm, '.2f',  'Matrice de confusion (normalisée)'),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=CLASS_NAMES_FR, yticklabels=CLASS_NAMES_FR,
        ax=ax, linewidths=0.5
    )
    ax.set_title(title)
    ax.set_xlabel('Prédit')
    ax.set_ylabel('Réel')
    ax.tick_params(axis='x', rotation=45)
    ax.tick_params(axis='y', rotation=0)

plt.tight_layout()
plt.show()

# Classes les plus difficiles
per_class_acc = cm.diagonal() / cm.sum(axis=1)
print('\nPrécision par classe :')
for name, acc in sorted(zip(CLASS_NAMES_FR, per_class_acc), key=lambda x: x[1]):
    bar = '█' * int(acc * 20)
    print(f'  {name:<25} {bar:<20} {acc:.1%}')

In [ ]:
# ── Visualisation des prédictions ─────────────────────────────────────────────
val_generator.reset()
images_batch, labels_batch = next(val_generator)
preds_batch = model.predict(images_batch, verbose=0)

n_show = 16
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle('Prédictions sur l\'ensemble de validation', fontsize=14)

idx_to_class = {v: k for k, v in val_generator.class_indices.items()}

for i, ax in enumerate(axes.flat):
    if i >= len(images_batch):
        ax.axis('off')
        continue
    img   = images_batch[i]
    true  = CLASS_NAMES_FR[CLASS_NAMES.index(idx_to_class[np.argmax(labels_batch[i])])]
    pred  = CLASS_NAMES_FR[CLASS_NAMES.index(idx_to_class[np.argmax(preds_batch[i])])]
    conf  = np.max(preds_batch[i])
    color = 'green' if true == pred else 'red'

    ax.imshow(img)
    ax.set_title(f'V: {true}\nP: {pred} ({conf:.0%})', color=color, fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.show()
print('Vert = correct | Rouge = erreur de classification')

In [ ]:
# ── Analyse des erreurs ───────────────────────────────────────────────────────
errors = np.where(y_pred != y_true)[0]
print(f'Nombre d\'erreurs : {len(errors)} / {len(y_true)} ({len(errors)/len(y_true):.1%})')

# Top confusions
print('\nTop 10 confusions (classe réelle → classe prédite) :')
confusion_pairs = []
for i in errors:
    confusion_pairs.append((CLASS_NAMES_FR[y_true[i]], CLASS_NAMES_FR[y_pred[i]]))

from collections import Counter
top_confusions = Counter(confusion_pairs).most_common(10)
for (real, pred), count in top_confusions:
    print(f'  {real} → {pred} : {count} fois')

---
## 💾 Partie 6 — Sauvegarde du modèle (optionnel)

In [ ]:
# ── Sauvegarde au format Keras natif (.keras) ─────────────────────────────────
model.save('flower_classifier_final.keras')
print('Modèle sauvegardé → flower_classifier_final.keras')

# Sauvegarde au format H5 (compatibilité TF1 / anciens systèmes)
model.save('flower_classifier_final.h5')
print('Modèle sauvegardé → flower_classifier_final.h5')

# Sauvegarde au format SavedModel (pour TF Serving, TFLite, etc.)
model.export('flower_classifier_savedmodel')
print('Modèle exporté   → flower_classifier_savedmodel/')

In [ ]:
# ── Chargement et prédiction sur une nouvelle image ───────────────────────────
loaded_model = keras.models.load_model('flower_classifier_final.keras')

def predict_flower(image_path, model, class_names_fr=CLASS_NAMES_FR):
    """Prédit la classe d'une image et affiche les 3 meilleures hypothèses."""
    img = keras_image.load_img(image_path, target_size=(HEIGHT, WIDTH))
    arr = keras_image.img_to_array(img) / 255.0
    arr = np.expand_dims(arr, axis=0)

    probs = loaded_model.predict(arr, verbose=0)[0]
    top3  = np.argsort(probs)[::-1][:3]

    plt.figure(figsize=(4, 4))
    plt.imshow(img)
    plt.title(f'{class_names_fr[top3[0]]} ({probs[top3[0]]:.1%})', fontsize=12)
    plt.axis('off')
    plt.show()

    print('Top 3 prédictions :')
    for i in top3:
        print(f'  {class_names_fr[i]:<25} {probs[i]:.1%}')

# Exemple d'utilisation :
# predict_flower('chemin/vers/votre/fleur.jpg', loaded_model)

# Test avec une image de validation
sample_val_cls  = CLASS_NAMES[0]
sample_val_dir  = os.path.join(VAL_DIR, sample_val_cls)
sample_val_file = os.path.join(sample_val_dir, os.listdir(sample_val_dir)[0])
predict_flower(sample_val_file, loaded_model)

### 🚀 Pistes de déploiement

| Option | Outil | Notes |
|---|---|---|
| **API REST locale** | Flask / FastAPI | `model.predict()` dans un endpoint POST |
| **Cloud** | Google Cloud Run, AWS Lambda | Conteneuriser avec Docker + modèle SavedModel |
| **Mobile** | TFLite | `tf.lite.TFLiteConverter.from_saved_model(...)` puis déploiement Android/iOS |
| **Web** | TensorFlow.js | `tensorflowjs_converter` puis exécution in-browser |

```python
# Conversion TFLite (exemple)
converter = tf.lite.TFLiteConverter.from_saved_model('flower_classifier_savedmodel')
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # Quantification INT8
tflite_model = converter.convert()
with open('flower_classifier.tflite', 'wb') as f:
    f.write(tflite_model)
```

---
## ✅ Récapitulatif

| Étape | Ce qui a été fait |
|---|---|
| **Données** | 13 642 images train + 98 val, 14 classes, redimensionnées à 48×48 |
| **Architecture** | 4 blocs Conv-BN-Pool + GAP + Dense-Dropout, ~1.5 M paramètres |
| **Hyperparamètres** | Adam lr=1e-3, batch=32, EarlyStopping + ReduceLROnPlateau |
| **Augmentation** | Rotation, flip, zoom, shift, shear, brightness |
| **Évaluation** | Accuracy/Loss courbes, rapport classification, matrice de confusion |
| **Sauvegarde** | `.keras`, `.h5`, SavedModel |

**Pour aller plus loin :** Transfer Learning avec MobileNetV2 ou EfficientNetB0 permettrait typiquement d'atteindre 85-95% de précision sur ce dataset.